# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/falah-bit/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Lane 2 — Refresh / Content Opportunity Scoring**, framed around an **early-warning signal for declining content**.

I'm choosing this lane because the starter dataset already ships an observable trend signal (`trend_direction`, `trend_pct`) plus a matching reason code in the reference pipeline (`declining_with_demand`), so the direction is well-supported by data I actually have. It also targets a real, familiar workflow gap: content and SEO teams typically notice a page has lost traffic only *after* the drop is already severe, rather than catching it early while it's still small and actionable. A ranked, evidence-backed "review this first" list is a concrete, useful output a reviewer can act on immediately.

I'm treating this as provisional — I can confirm or adjust it through Week 4, per the lane guide.

In [28]:
# Quick check: confirm the lane's core signal exists and get a first sense of its scale.
import pandas as pd
import os

possible_paths = [
    'data/raw/content_refresh_anonymized.csv',
    '../../data/raw/content_refresh_anonymized.csv',
    '../data/raw/content_refresh_anonymized.csv',
]
csv_path = next((p for p in possible_paths if os.path.exists(p)), None)
if csv_path is None:
    raise FileNotFoundError(
        "CSV not found in expected locations — clone the repo first if running in a fresh Colab session:\n"
        "!git clone https://github.com/falah-bit/flyrank-ml-internship.git"
    )

df = pd.read_csv(csv_path)
print(f"Loaded: {csv_path}")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
print("trend_direction values:", df['trend_direction'].unique())

Loaded: data/raw/content_refresh_anonymized.csv
Rows: 30,000 | Columns: 44
trend_direction values: ['down' 'stable' 'new' 'up' 'flat']


## 2. The question: decision, action, cost of a wrong call

**Decision this work improves:** which content pages should be reviewed first for a refresh, out of a much larger set no reviewer has time to check individually.

**Who acts on it:** an SEO/content reviewer or content strategist with limited weekly review capacity — they read the ranked queue and start from the top.

**Cost of a wrong call:**
- **False negative** (a genuinely declining page not flagged): the page keeps losing impressions and clicks for weeks before anyone notices — the loss compounds the longer it goes unreviewed.
- **False positive** (a stable page flagged as declining): reviewer time is spent investigating a page that didn't need attention, at the expense of a page that did.

Because review capacity is limited, a **ranking** (not just a yes/no label) is what actually matters here — this is why it fits Lane 2's ranked-queue framing rather than a plain classifier.

In [29]:
# Make the "cost of missing one" concrete: how much traffic is actually at stake
# on pages already showing a decline, versus pages that aren't?
df_valid = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)]

avg_imp_declining = df_valid.loc[df_valid['trend_direction'] == 'down', 'impressions_90d'].mean()
avg_imp_stable = df_valid.loc[df_valid['trend_direction'] != 'down', 'impressions_90d'].mean()

print(f"Avg 90d impressions on declining pages:     {avg_imp_declining:.0f}")
print(f"Avg 90d impressions on non-declining pages: {avg_imp_stable:.0f}")


Avg 90d impressions on declining pages:     4919
Avg 90d impressions on non-declining pages: 5533


## 3. Quick look at the data (2-3 real numbers)

*(Run the cell below first, then copy the printed numbers into this paragraph before committing — the numbers below are placeholders.)*

After filtering to valid rows (`impressions_90d > 0` and `content_age_days >= 90`), **[pct_declining]%** of pages are currently flagged `trend_direction == 'down'`, out of **[n_valid_rows]** valid rows across **[n_clients]** unique clients. The median 90-day impression volume on those rows is **[median_impressions]**, which confirms these aren't low-traffic edge cases — real visibility is on the line. That combination (a non-trivial share of pages, real traffic volume, across many clients) is why this looks worth the next 7 weeks.

In [30]:
pct_declining = (df_valid['trend_direction'] == 'down').mean() * 100
median_impressions = df_valid['impressions_90d'].median()
n_clients = df_valid['client_id'].nunique()
n_valid_rows = len(df_valid)

print(f"Valid rows after filtering:      {n_valid_rows:,}")
print(f"% of pages flagged 'declining':  {pct_declining:.1f}%")
print(f"Median 90-day impressions:       {median_impressions}")
print(f"Unique clients represented:      {n_clients}")

Valid rows after filtering:      30,000
% of pages flagged 'declining':  54.2%
Median 90-day impressions:       731.0
Unique clients represented:      32


## 4. Careful words: what I can and can't claim

**What I can claim:**
- An **observed** pattern: a measurable share of pages currently show a downward trend signal in this window.
- A **directional** signal that can help prioritize which pages a human reviews first.
- A **decision-support** output — a ranked queue with reason codes for a reviewer to inspect, not an automatic decision.

**What I cannot claim:**
- **Causal proof.** I can't say a refresh *caused* a recovery, or that any factor *caused* the decline, without a real experiment.
- **"Predicting Google."** I'm not reverse-engineering or predicting the search algorithm — only observing this dataset's own historical signals.
- That the starter label is a future outcome. `is_declining_label` (`trend_direction == 'down'`) is a **proxy computed from the current window**, not a forward-looking result — it's a starting point, not the final target. A stronger version of this lane, once I move to the warehouse release, should define the label from a *future* window (e.g. features from the prior 90 days predicting decline over the next 30) rather than the current one.
- That every drop is a real decline. Before treating a page as declining I still need to rule out look-alikes — consolidation (a sibling page absorbing the traffic), seasonality, and simple low-volume noise.

In [31]:
# No computation needed for this section — the claims above are qualitative,
# grounded in the numbers already computed in Sections 2-3.
print("Section 4: no additional computation required.")

Section 4: no additional computation required.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.